In [28]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [29]:
import pandas as pd

In [30]:
# 데이터 불러오기
df = pd.read_csv('/content/drive/MyDrive/2025_데이터마이닝/WiDS/training.csv')
df_test = pd.read_csv('/content/drive/MyDrive/2025_데이터마이닝/WiDS/test.csv')

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12906 entries, 0 to 12905
Data columns (total 83 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   patient_id                             12906 non-null  int64  
 1   patient_race                           6521 non-null   object 
 2   payer_type                             11103 non-null  object 
 3   patient_state                          12855 non-null  object 
 4   patient_zip3                           12906 non-null  int64  
 5   patient_age                            12906 non-null  int64  
 6   patient_gender                         12906 non-null  object 
 7   bmi                                    3941 non-null   float64
 8   breast_cancer_diagnosis_code           12906 non-null  object 
 9   breast_cancer_diagnosis_desc           12906 non-null  object 
 10  metastatic_cancer_diagnosis_code       12906 non-null  object 
 11  me

### **breast_cancer_diagnosis_code**  (유방암 진단 코드)

In [32]:
# breast_cancer_diagnosis_code 값별 등장 횟수
code_counts = df['breast_cancer_diagnosis_code'].value_counts(dropna=False)

# 비율 계산
code_percent = (code_counts / len(df)) * 100

# 합치기
code_summary = pd.DataFrame({
    'Count': code_counts,
    'Percent': code_percent.round(2)
})

code_summary = code_summary.sort_values(by='Count', ascending=False)

print("breast_cancer_diagnosis_code 분포 요약 (Convert 전):")
display(code_summary)

breast_cancer_diagnosis_code 분포 요약 (Convert 전):


,Count,Percent
breast_cancer_diagnosis_code,,
1749,1982,15.36
C50911,1797,13.92
C50912,1712,13.27
C50919,1467,11.37
C50411,978,7.58
C50412,877,6.80
C50811,491,3.80
C50812,419,3.25
1744,389,3.01


In [33]:
# # Convert ICD-9 -> ICD-10
# replace_mapping = {
#     '1749': 'C50919',
#     '1744': 'C50419',
#     '1748': 'C50819',
#     '1742': 'C50219',
#     '1741': 'C50119',
#     '1745': 'C50519',
#     '1743': 'C50319',
#     '1746': 'C50619',
#     '19881': 'C7981',
#     '1759': 'C50929'
# }

# # 매핑 적용
# df['breast_cancer_diagnosis_code'] = df['breast_cancer_diagnosis_code'].replace(replace_mapping)

In [34]:
# breast_cancer_diagnosis_code 값별 등장 횟수
code_counts = df['breast_cancer_diagnosis_code'].value_counts(dropna=False)

# 비율 계산
code_percent = (code_counts / len(df)) * 100

# 합치기
code_summary = pd.DataFrame({
    'Count': code_counts,
    'Percent': code_percent.round(2)
})

code_summary = code_summary.sort_values(by='Count', ascending=False)

print("breast_cancer_diagnosis_code 분포 요약 (Convert 후):")
display(code_summary)

breast_cancer_diagnosis_code 분포 요약 (Convert 후):


,Count,Percent
breast_cancer_diagnosis_code,,
1749,1982,15.36
C50911,1797,13.92
C50912,1712,13.27
C50919,1467,11.37
C50411,978,7.58
C50412,877,6.80
C50811,491,3.80
C50812,419,3.25
1744,389,3.01


In [35]:
# 코드별로 target 값 분포(count)
code_target_crosstab = pd.crosstab(df['breast_cancer_diagnosis_code'], df['DiagPeriodL90D'])

# target 1 비율 계산
code_target_crosstab['target_1_rate'] = code_target_crosstab[1] / (code_target_crosstab[0] + code_target_crosstab[1])

# 비율 높은 순으로 정렬
code_target_crosstab = code_target_crosstab.sort_values(by='target_1_rate', ascending=False)

print("🧠 breast_cancer_diagnosis_code별 타겟 분포 + 전이율 요약:")
display(code_target_crosstab)

🧠 breast_cancer_diagnosis_code별 타겟 분포 + 전이율 요약:


DiagPeriodL90D,0,1,target_1_rate
breast_cancer_diagnosis_code,,,
C5011,0,2,1.000000
C50021,0,1,1.000000
C50,0,1,1.000000
C5051,0,1,1.000000
C5031,0,1,1.000000
C509,0,3,1.000000
C50619,0,3,1.000000
C5021,0,3,1.000000
C5081,1,7,0.875000


In [36]:
# 19881 코드 가진 사람들 필터링
secondary_breast = df[df['breast_cancer_diagnosis_code'] == '19881']

# metastatic 관련 칼럼들 + 타겟 확인
# metastatic_cancer_diagnosis_code, metastatic_first_novel_treatment, metastatic_first_novel_treatment_type, DiagPeriodL90D
columns_to_check = [
    'breast_cancer_diagnosis_desc',
    'metastatic_cancer_diagnosis_code',
    'metastatic_first_novel_treatment',
    'metastatic_first_novel_treatment_type',
    'DiagPeriodL90D'
]

# 결과 출력
print("🔍 19881 코드 환자들의 metastatic 관련 정보 + 타겟 값 확인:")
display(secondary_breast[columns_to_check])

🔍 19881 코드 환자들의 metastatic 관련 정보 + 타겟 값 확인:


,breast_cancer_diagnosis_desc,metastatic_cancer_diagnosis_code,metastatic_first_novel_treatment,metastatic_first_novel_treatment_type,DiagPeriodL90D
1252,Secondary malignant neoplasm of breast,C7981,NaN,NaN,0
4868,Secondary malignant neoplasm of breast,C7951,NaN,NaN,0
5429,Secondary malignant neoplasm of breast,C7981,NaN,NaN,0
7652,Secondary malignant neoplasm of breast,C779,NaN,NaN,0
9523,Secondary malignant neoplasm of breast,C7981,NaN,NaN,0
10328,Secondary malignant neoplasm of breast,C7981,NaN,NaN,1
10595,Secondary malignant neoplasm of breast,C7981,NaN,NaN,0
12325,Secondary malignant neoplasm of breast,C7951,NaN,NaN,0
12650,Secondary malignant neoplasm of breast,C7981,NaN,NaN,0


In [37]:
import numpy as np

# 1. ICD-9/ICD-10 구분하는 새 컬럼 만들기
df['ICD_version'] = np.where(df['breast_cancer_diagnosis_code'].str.isdigit(), 'ICD-9', 'ICD-10')

# 2. ICD-9 vs ICD-10 그룹별로 타겟별(0,1) 환자 수 세기
icd_target_counts = pd.crosstab(df['ICD_version'], df['DiagPeriodL90D'])

# 3. ICD-9 vs ICD-10 그룹별 전이율(mean) 계산
icd_target_means = df.groupby('ICD_version')['DiagPeriodL90D'].mean()

# 4. 둘 다 보기 좋게 합치기
icd_summary = icd_target_counts.copy()
icd_summary['target_1_rate'] = icd_target_means

# 결과 출력
print("🎯 ICD-9 vs ICD-10 환자들의 타겟(0/1) 분포 + 전이율 비교")
display(icd_summary)

🎯 ICD-9 vs ICD-10 환자들의 타겟(0/1) 분포 + 전이율 비교


DiagPeriodL90D,0,1,target_1_rate
ICD_version,,,
ICD-10,2139,7776,0.784266
ICD-9,2707,284,0.094952


**-> breast_cancer_code가 ICD-9인 행들의 전이율이 매우 낮음**

### **breast_cancer_diagnosis_desc** (유방암 진단 설명)

In [38]:
## 성별(female/male), 방향(right/left), unspecified 여부, 부위(upper/lower/central 등) 일치 여부

# 1. 성별 일치 여부
def check_gender(row):
    desc = str(row['breast_cancer_diagnosis_desc']).lower()
    code = row['breast_cancer_diagnosis_code']

    if 'female' in desc:
        return 'female'
    elif 'male' in desc:
        return 'male'
    else:
        return 'unknown'

# 2. 방향(right/left) 일치 여부
def check_side(row):
    desc = str(row['breast_cancer_diagnosis_desc']).lower()

    if 'right' in desc:
        return 'right'
    elif 'left' in desc:
        return 'left'
    else:
        return 'unknown'

# 3. unspecified 여부 (site 불명확 여부)
def check_unspecified(row):
    desc = str(row['breast_cancer_diagnosis_desc']).lower()
    return 'unspecified' if 'unspecified' in desc else 'specified'

# 4. 부위 키워드 대략 매칭 (optional)
def check_area(row):
    desc = str(row['breast_cancer_diagnosis_desc']).lower()

    if 'upper-outer' in desc:
        return 'upper-outer'
    elif 'upper-inner' in desc:
        return 'upper-inner'
    elif 'lower-outer' in desc:
        return 'lower-outer'
    elif 'lower-inner' in desc:
        return 'lower-inner'
    elif 'central' in desc:
        return 'central'
    elif 'nipple' in desc or 'areola' in desc:
        return 'nipple/areola'
    else:
        return 'other/unknown'

# 이제 적용!
df['desc_gender'] = df.apply(check_gender, axis=1)
df['desc_side'] = df.apply(check_side, axis=1)
df['desc_unspecified'] = df.apply(check_unspecified, axis=1)
df['desc_area'] = df.apply(check_area, axis=1)

# 결과 확인
print("🔍 Generated comparison columns:")
display(df[['breast_cancer_diagnosis_code', 'breast_cancer_diagnosis_desc', 'desc_gender', 'desc_side', 'desc_unspecified', 'desc_area']].head(10))

🔍 Generated comparison columns:


,breast_cancer_diagnosis_code,breast_cancer_diagnosis_desc,desc_gender,desc_side,desc_unspecified,desc_area
0,C50919,Malignant neoplasm of unsp site of unspecified...,female,unknown,unspecified,other/unknown
1,C50411,Malig neoplm of upper-outer quadrant of right ...,female,right,specified,upper-outer
2,C50112,Malignant neoplasm of central portion of left ...,female,left,specified,central
3,C50212,Malig neoplasm of upper-inner quadrant of left...,female,left,specified,upper-inner
4,1749,"Malignant neoplasm of breast (female), unspeci...",female,unknown,unspecified,other/unknown
5,1749,"Malignant neoplasm of breast (female), unspeci...",female,unknown,unspecified,other/unknown
6,C50912,Malignant neoplasm of unspecified site of left...,female,left,unspecified,other/unknown
7,C50512,Malig neoplasm of lower-outer quadrant of left...,female,left,specified,lower-outer
8,1744,Malignant neoplasm of upper-outer quadrant of ...,female,unknown,specified,upper-outer
9,C50912,Malignant neoplasm of unspecified site of left...,female,left,unspecified,other/unknown


In [39]:
df["is_female"] = df.breast_cancer_diagnosis_desc.str.contains("female").astype("int")
df["is_female"].value_counts()

,count
is_female,
1,12887
0,19


In [40]:
# is_female == 0인 데이터만 필터링
not_female = df[df['is_female'] == 0]

# 그 안에서 patient_gender 확인
gender_check = not_female['patient_gender'].value_counts()
print("🔍 is_female == 0 인 사람들의 patient_gender 분포:")
display(gender_check)

🔍 is_female == 0 인 사람들의 patient_gender 분포:


,count
patient_gender,
F,19


In [41]:
# 코드별로 desc 고유값 개수 세기
code_desc_unique_counts = df.groupby('breast_cancer_diagnosis_code')['breast_cancer_diagnosis_desc'].nunique()

# 고유 desc가 1개 초과하는 코드만 보기 (즉, 코드 하나에 여러 desc가 있는 경우)
inconsistent_desc = code_desc_unique_counts[code_desc_unique_counts > 1]

print("🔍 코드별 desc 고유값 개수 요약:")
display(code_desc_unique_counts)

print("\n🚨 desc가 여러 개인 코드들:")
display(inconsistent_desc)

🔍 코드별 desc 고유값 개수 요약:


,breast_cancer_diagnosis_desc
breast_cancer_diagnosis_code,
1741,1
1742,1
1743,1
1744,1
1745,1
1746,1
1748,1
1749,1
1759,1



🚨 desc가 여러 개인 코드들:


,breast_cancer_diagnosis_desc
breast_cancer_diagnosis_code,


In [42]:
# desc별로 그룹화
desc_groups = df.groupby('breast_cancer_diagnosis_desc')

# 그룹별 몇 명씩 있는지 먼저 확인
desc_counts = desc_groups.size().sort_values(ascending=False)

print("🔍 각 desc 그룹별 환자 수:")
display(desc_counts)

🔍 각 desc 그룹별 환자 수:


,0
breast_cancer_diagnosis_desc,
"Malignant neoplasm of breast (female), unspecified",1982
Malignant neoplasm of unsp site of right female breast,1797
Malignant neoplasm of unspecified site of left female breast,1712
Malignant neoplasm of unsp site of unspecified female breast,1467
Malig neoplm of upper-outer quadrant of right female breast,978
Malig neoplasm of upper-outer quadrant of left female breast,877
Malignant neoplasm of ovrlp sites of right female breast,491
Malignant neoplasm of ovrlp sites of left female breast,419
Malignant neoplasm of upper-outer quadrant of female breast,389


In [43]:
# 1. desc별로 그룹화
desc_groups = df.groupby('breast_cancer_diagnosis_desc')

# 2. 주요 특징 요약하기 (patient_age 사용)
desc_summary = desc_groups.agg({
    'DiagPeriodL90D': ['mean', 'count'],  # 전이율 평균, 환자 수
    'patient_age': 'mean',                # 평균 나이
    'income_individual_median': 'mean',                 # 평균 소득
    'female': 'mean',                  # 여성 비율
    'metastatic_cancer_diagnosis_code': pd.Series.nunique  # 서로 다른 전이 부위 코드 종류 수
})

# 3. 컬럼 이름 깔끔하게 다듬기
desc_summary.columns = ['_'.join(col).strip() for col in desc_summary.columns.values]

# 4. 결과 보기
print("🔍 desc 그룹별 주요 특징 요약 (patient_age 기준):")
display(desc_summary.sort_values('DiagPeriodL90D_count', ascending=False))

🔍 desc 그룹별 주요 특징 요약 (patient_age 기준):


,DiagPeriodL90D_mean,DiagPeriodL90D_count,patient_age_mean,income_individual_median_mean,female_mean,metastatic_cancer_diagnosis_code_nunique
breast_cancer_diagnosis_desc,,,,,,
"Malignant neoplasm of breast (female), unspecified",0.091322,1982,59.739657,36321.459032,49.954915,28
Malignant neoplasm of unsp site of right female breast,0.788536,1797,59.283250,36431.300053,49.867581,30
Malignant neoplasm of unspecified site of left female breast,0.776869,1712,59.904206,36557.047278,49.845837,33
Malignant neoplasm of unsp site of unspecified female breast,0.780504,1467,60.519427,36971.297347,49.959430,35
Malig neoplm of upper-outer quadrant of right female breast,0.783231,978,57.958078,36188.822709,49.883670,25
Malig neoplasm of upper-outer quadrant of left female breast,0.797035,877,58.873432,36792.199556,49.805986,23
Malignant neoplasm of ovrlp sites of right female breast,0.828921,491,59.588595,36259.535568,49.891134,20
Malignant neoplasm of ovrlp sites of left female breast,0.792363,419,58.482100,36232.203268,49.943132,17
Malignant neoplasm of upper-outer quadrant of female breast,0.087404,389,56.863753,35658.349824,49.877145,20


### **metastatic_first_novel_treatment** (전이 후 첫 치료)
+ **metastatic_first_novel_treatment_type** (전이 후 첫 치료 유형 설명)

In [44]:
# 1. metastatic_first_novel_treatment 고유값, 결측치 확인
print("== metastatic_first_novel_treatment 고유값: ==")
print(df['metastatic_first_novel_treatment'].value_counts(dropna=False))

# 2. metastatic_first_novel_treatment_type 고유값, 결측치 확인
print("\n== metastatic_first_novel_treatment_type 고유값: ==")
print(df['metastatic_first_novel_treatment_type'].value_counts(dropna=False))

== metastatic_first_novel_treatment 고유값: ==
metastatic_first_novel_treatment
NaN              12882
PEMBROLIZUMAB       13
OLAPARIB            11
Name: count, dtype: int64

== metastatic_first_novel_treatment_type 고유값: ==
metastatic_first_novel_treatment_type
NaN                12882
Antineoplastics       24
Name: count, dtype: int64


***→ 고유값 개수 동일***

- **PEMBROLIZUMAB** : 면역항암 치료제인 PD-1 억제제로, 암세포가 면역체계를 피하는 것을 막아 면역세포가 암세포를 공격하도록 도움


- **OLAPARIB** : PARP 억제제로서, 암세포를 죽이는 데 사용되는 표적 항암제


- **Antineoplastics** : 항암제


In [45]:
# 1. metastatic_first_novel_treatment 값별 전이율 + 샘플 수
print("metastatic_first_novel_treatment별 DiagPeriodL90D 전이율 + 개수")
treatment_summary = df.groupby('metastatic_first_novel_treatment')['DiagPeriodL90D'].agg(['mean', 'count'])
display(treatment_summary)

# 2. metastatic_first_novel_treatment_type 값별 전이율 + 샘플 수
print("\nmetastatic_first_novel_treatment_type별 DiagPeriodL90D 전이율 + 개수")
treatment_type_summary = df.groupby('metastatic_first_novel_treatment_type')['DiagPeriodL90D'].agg(['mean', 'count'])
display(treatment_type_summary)

metastatic_first_novel_treatment별 DiagPeriodL90D 전이율 + 개수


,mean,count
metastatic_first_novel_treatment,,
OLAPARIB,0.545455,11
PEMBROLIZUMAB,0.769231,13



metastatic_first_novel_treatment_type별 DiagPeriodL90D 전이율 + 개수


,mean,count
metastatic_first_novel_treatment_type,,
Antineoplastics,0.666667,24


In [46]:
# treatment 여부에 따라 patient_age, income_mean, bmi 비교
display(df.groupby('metastatic_first_novel_treatment')[['patient_age', 'income_individual_median', 'bmi']].mean())

,patient_age,income_individual_median,bmi
metastatic_first_novel_treatment,,,
OLAPARIB,44.545455,34788.761859,33.95
PEMBROLIZUMAB,57.538462,38569.354928,31.93


In [47]:
# treatment_type별로 patient_age, income_mean 비교
display(df.groupby('metastatic_first_novel_treatment_type')[['patient_age', 'income_individual_median']].mean())

,patient_age,income_individual_median
metastatic_first_novel_treatment_type,,
Antineoplastics,51.583333,36836.583105


In [48]:
# 둘 다 값이 있는 사람만 필터링
non_null_both = df[
    df['metastatic_first_novel_treatment'].notna() &
    df['metastatic_first_novel_treatment_type'].notna()
]

# 원하는 네 컬럼 출력
print(f"🔎 치료 관련 칼럼과 income_individual_median, patient_age 출력 (데이터 수: {len(non_null_both)})")
display(non_null_both[['metastatic_first_novel_treatment', 'metastatic_first_novel_treatment_type', 'income_individual_median', 'patient_age']])

🔎 치료 관련 칼럼과 income_individual_median, patient_age 출력 (데이터 수: 24)


,metastatic_first_novel_treatment,metastatic_first_novel_treatment_type,income_individual_median,patient_age
40,OLAPARIB,Antineoplastics,27208.76190,43
107,OLAPARIB,Antineoplastics,27712.36000,47
249,OLAPARIB,Antineoplastics,35253.08772,42
1386,OLAPARIB,Antineoplastics,37965.45833,59
2025,PEMBROLIZUMAB,Antineoplastics,33411.92683,67
2323,PEMBROLIZUMAB,Antineoplastics,44924.00000,75
2379,PEMBROLIZUMAB,Antineoplastics,28834.55172,68
2647,OLAPARIB,Antineoplastics,38888.60417,34
3515,OLAPARIB,Antineoplastics,32849.60417,32
3925,OLAPARIB,Antineoplastics,28367.83333,38


In [49]:
# 1. 둘 다 값이 있는 사람만 필터링
non_null_both = df[
    df['metastatic_first_novel_treatment'].notna() &
    df['metastatic_first_novel_treatment_type'].notna()
]

# 2. 필요한 평균 계산
subset_means = non_null_both[['income_individual_median', 'patient_age']].mean()
overall_means = df[['income_individual_median', 'patient_age']].mean()

# 3. 비교 테이블 만들기
comparison = pd.DataFrame({
    'subset_mean': subset_means,
    'overall_mean': overall_means
})

# 결과 출력
print("🎯 치료 정보가 있는 환자 vs 전체 환자 평균 비교 (소득 & 나이)")
display(comparison)

🎯 치료 정보가 있는 환자 vs 전체 환자 평균 비교 (소득 & 나이)


,subset_mean,overall_mean
income_individual_median,36836.583105,36520.522622
patient_age,51.583333,59.183326


In [50]:
# 1. 치료 정보 둘 다 값이 있는 사람
non_null_both = df[
    df['metastatic_first_novel_treatment'].notna() &
    df['metastatic_first_novel_treatment_type'].notna()
]

# 2. 치료 정보 없는 사람
null_both = df[
    df['metastatic_first_novel_treatment'].isna() &
    df['metastatic_first_novel_treatment_type'].isna()
]

# 3. 각각 평균 계산
subset_means = non_null_both[['income_individual_median', 'patient_age']].mean()
null_means = null_both[['income_individual_median', 'patient_age']].mean()

# 4. 비교 테이블 만들기
comparison = pd.DataFrame({
    'treatment_info_present': subset_means,
    'treatment_info_absent': null_means
})

# 결과 출력
print("🎯 치료 정보 '있음' vs '없음' 환자 평균 비교 (소득 & 나이)")
display(comparison)

🎯 치료 정보 '있음' vs '없음' 환자 평균 비교 (소득 & 나이)


,treatment_info_present,treatment_info_absent
income_individual_median,36836.583105,36519.933735
patient_age,51.583333,59.197485


**→ 치료 칼럼들 제외한 이유가 있음**

> 값이 있는 데이터가 24개뿐이라서, 혹시 이 사람들이 부유한 사람들인가 ? (치료 받았으니) 싶었는데 !

> 또 그런건 아님 .. 평균 (소득, 나이) 비교해보니 비슷해 ~ .. 의미없나봄

### 추가

In [51]:
df["is_female"] = df.breast_cancer_diagnosis_desc.str.contains("female").astype("int")
df["is_female"].value_counts()

,count
is_female,
1,12887
0,19


In [52]:
#  Find Product of Pollutants
df["N02xOzonexPM25"]=df["N02"]*df["Ozone"]*df["PM25"]

In [53]:
# Drop some features
# List of columns to iterate over
columns_to_iterate = [col for col in df.columns if col not in ["patient_zip3", "N02xOzonexPM25"]]

# Iterate over each column
for col in columns_to_iterate:
    # Your code to operate on each column goes here
    df["check"]=df.groupby(["patient_zip3","N02xOzonexPM25"])[col].transform("nunique")
    if df["check"].max()==1:
        print("dropped ",col)
        df=df.drop(col,axis=1)
df=df.drop("check",axis=1)

dropped  patient_gender
dropped  metastatic_first_novel_treatment
dropped  metastatic_first_novel_treatment_type
dropped  population
dropped  density
dropped  age_median
dropped  age_under_10
dropped  age_10_to_19
dropped  age_20s
dropped  age_30s
dropped  age_40s
dropped  age_50s
dropped  age_60s
dropped  age_70s
dropped  age_over_80
dropped  male
dropped  female
dropped  married
dropped  divorced
dropped  never_married
dropped  widowed
dropped  family_size
dropped  family_dual_income
dropped  income_household_median
dropped  income_household_under_5
dropped  income_household_5_to_10
dropped  income_household_10_to_15
dropped  income_household_15_to_20
dropped  income_household_20_to_25
dropped  income_household_25_to_35
dropped  income_household_35_to_50
dropped  income_household_50_to_75
dropped  income_household_75_to_100
dropped  income_household_100_to_150
dropped  income_household_150_over
dropped  income_household_six_figure
dropped  income_individual_median
dropped  home_owner

In [54]:
for col in df.columns:
    print(col)

patient_id
patient_race
payer_type
patient_state
patient_zip3
patient_age
bmi
breast_cancer_diagnosis_code
breast_cancer_diagnosis_desc
metastatic_cancer_diagnosis_code
Region
Division
DiagPeriodL90D
ICD_version
desc_gender
desc_side
desc_unspecified
desc_area
is_female
N02xOzonexPM25
